In [19]:
import torch
import torchvision
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

In [20]:
path = kagglehub.dataset_download("waalbannyantudre/hate-speech-detection-curated-dataset")

In [21]:
print(path)

C:\Users\gauma\.cache\kagglehub\datasets\waalbannyantudre\hate-speech-detection-curated-dataset\versions\1


In [22]:
import os
hate_speech_data = pd.read_csv(os.path.join(path, "HateSpeechDatasetBalanced.csv"))
print(hate_speech_data.head())
print(hate_speech_data['Label'].value_counts())

                                             Content  Label
0  denial of normal the con be asked to comment o...      1
1  just by being able to tweet this insufferable ...      1
2  that is retarded you too cute to be single tha...      1
3  thought of a real badass mongol style declarat...      1
4                                afro american basho      1
Label
1    364525
0    361594
Name: count, dtype: int64


In [23]:
import nltk, spacy, re, string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer, WordNetLemmatizer
import contractions

nltk.download(['wordnet', 'punkt', 'stopwords'], quiet=True)

stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

punctuation_pattern = re.compile(f"[{re.escape(string.punctuation)}]")
digit_pattern = re.compile(r'\d+')
whitespace_pattern = re.compile(r'\s+')
non_word_pattern = re.compile(r'\W+')

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    
    text = text.lower()
    text = contractions.fix(text)
    text = digit_pattern.sub(' ', text)
    text = punctuation_pattern.sub(' ', text)
    text = non_word_pattern.sub(' ', text)
    text = whitespace_pattern.sub(' ', text).strip()
    tokens = word_tokenize(text)
    res = []
    for token in tokens:
        if token not in stop_words and len(token) < 50:
            processed_token = stemmer.stem(token)
            res.append(processed_token)
    res_full = " ".join(res)
    return res_full
    
hate_speech_data['Cleaned_Content'] = hate_speech_data['Content'].apply(clean_text)

In [40]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(hate_speech_data, test_size=0.2, random_state=42, shuffle=True, stratify=hate_speech_data['Label'])

In [41]:
len(train_data), len(test_data)

(580895, 145224)

In [66]:
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import DistilBertModel, DistilBertTokenizer

from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding


class HateSpeechDataset(Dataset):
    
    def __init__(self, data):
        # Initialize BERT tokenizer
        self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')  
        enc = self.tokenizer(
            text=data["Cleaned_Content"].tolist(),
            truncation=True,
            padding="max_length",
            max_length=64
        )

        self.input_ids = enc["input_ids"]
        self.attn = enc["attention_mask"]
        self.labels = data["Label"].tolist()      
        self.data = data
       
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        '''
        example = self.data.iloc[idx]

        text = example["Cleaned_Content"]
        label = example["Label"]

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(text, padding='max_length', truncation=True, max_length=64, return_tensors='pt')
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"], ##.unsqueeze(0).int(),
            "label": label,
        }
        '''
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attn[idx]),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }
    

In [67]:


tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')      



Lemmatize data

In [68]:

class HateSpeechClassifier(nn.Module):
    def __init__(self):
        super(HateSpeechClassifier, self).__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.avgpool = nn.AdaptiveMaxPool1d(1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        outputs = self.avgpool(outputs.transpose(1, 2)).squeeze(-1)
        output = self.classifier(self.dropout(outputs))
        return output

In [69]:
train_dataset = HateSpeechDataset(train_data)
test_dataset = HateSpeechDataset(test_data)

from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer=tokenizer)
batch_size = 32
dataloader_train = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collator, shuffle=True)
dataloader_test = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collator, shuffle=False)   
model = HateSpeechClassifier()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [62]:
thing = enumerate(tqdm(dataloader_train))
thing

In [ ]:
from tqdm import tqdm
train_losses = []
test_losses = []
num_epochs = 3

for epoch in range(num_epochs):
    print(f"epoch {epoch+1}/{num_epochs}")
    model.train()
    total_train_loss = 0
    for i, batch in enumerate(tqdm(dataloader_train)):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.squeeze(), labels.float())
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(dataloader_train)
    train_losses.append(avg_train_loss)

    model.eval()
    total_test_loss = 0
    preds = list()
    truths = list()
    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader_test)):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.squeeze(), labels.float())
            total_test_loss += loss.item()
    avg_test_loss = total_test_loss / len(dataloader_test)
    test_losses.append(avg_test_loss)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss}, Test Loss: {avg_test_loss}")


epoch 1/3


batch 1/18153


batch 2/18153


batch 3/18153


batch 4/18153


batch 5/18153


batch 6/18153


batch 7/18153


  0%|          | 6/18153 [02:12<111:02:50, 22.03s/it]


KeyboardInterrupt: 